# Figure 2: Clonal architecture and public convergence of T cell repertoires

This notebook reproduces panels of **Figure 2** of the AIDA AIRR manuscript:

- **Fig. 2A** — UMAP of T cells with productive paired TCR αβ chains used for downstream analysis
- **Fig. 2B** — Distribution of TCR clonality (Gini coefficient) across T cell subsets
- **Fig. 2C** — Correlation of rarefied TCR Gini coefficient with donor age in CD4+, CD8+, and clonally expanded subsets
- **Fig. 2D** — UMAP overlay of public clones, colored by number of ethnicities sharing each clonotype
- **Fig. 2E** — Stacked bar plot of public clone proportions per T cell subset
- **Fig. 2F** — Proportions of TCRs with predicted epitope specificity (VDJdb) by sharing breadth
- **Fig. 2G** — Representative public clones (logo + clone passport)


## Imports and global plotting settings

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import os, ast, re
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats as stats
from scipy.stats import spearmanr, linregress, mannwhitneyu, kruskal, norm
from scipy.sparse import csr_matrix

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as path_effects
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch, PathPatch
from matplotlib.path import Path
import matplotlib as mpl
import seaborn as sns

import scanpy as sc
import scanpy.external as sce
import anndata as ad
import dandelion as ddl
import scirpy as ir
import sceleto2 as scjp
import logomaker

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

%matplotlib inline
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, color_map='OrRd')

plt.rcParams['pdf.fonttype'] = 42
sns.set_style('ticks', {'axes.edgecolor': 'black', 'axes.edgewidth': 2})
sns.set_context("paper", font_scale=1.3, rc={'patch.linewidth': 1})

mpl.rcParams['font.family'] = 'Liberation Sans'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42


## Color palettes and category orders

In [ ]:
Ethnicity_order = ['Chinese', 'Malay', 'Indian', 'Japanese', 'Korean', 'Thai']
Ethnicity_colors = ["tomato", 'gold', "dodgerblue", "limegreen", "aquamarine", "orchid", 'gray']
sex_colors = ["#87CEFA", "#FB7C7C"]

T_anno2_order = [
    'T_CD4_Naive_SOX4', 'T_CD4_Naive', 'T_CD4_cTfh', 'T_CD4_Th1', 'T_CD4_Th2',
    'T_CD4_Th17', 'T_CD4_activated', 'T_CD4_CTL', 'T_CD4_Treg', 'T_CD4_IFN',
    'T_CD8_Naive_SOX4', 'T_CD8_Naive', 'T_CD8_TEM_GZMK', 'T_CD8_TEM_GZMB', 'T_CD8_KIR',
    'T_unc_MAIT', 'T_unc_gdT', 'T_unc_dnT', 'NK_CD56br', 'NK_CD56dim', 'NK_CD56dim_KLRC2'
]
T_anno2_colors = [
    "lightsteelblue", "#4a6fe3", "mediumorchid", "#bb7784", "darkorange",
    "#023fa5", "lightblue", "darkgreen", "#d6bcc0", "tan",
    "goldenrod", "yellowgreen", "lightcoral", "#d33f6a", "#11c638",
    "darkorchid", "#ef9708", "#0fcfc0", "#9cded6", "#f0b98d", "#f3e1eb",
]


# Fig. 2A — UMAP of T cells with paired TCR αβ chains

Load the AnnData containing T cells with their scirpy/dandelion TCR annotation and clone IDs.

In [ ]:
vdata = sc.read('data/06_250601_TDATA_scirpy_merging_meta_with_clone_id_Pt.h5ad')

In [ ]:
vdata = vdata[vdata.obs['Ethnicity']!='European']
vdata = vdata[~vdata.obs['Ethnicity'].isna()]

In [ ]:
meta = vdata.obs.drop_duplicates('PatientID').set_index('PatientID')[['Age','Sex','Ethnicity']]

In [ ]:
vdata.obs['anno2'] = vdata.obs['anno2'].cat.reorder_categories(T_anno2_order)
vdata.uns['anno2_colors'] = T_anno2_colors


In [ ]:
scjp.us(vdata,'anno2', frameon=False, legend_loc='on data',
        legend_fontsize=9, size=0.5)
plt.title('')
scjp.save_fig('Fig2_','conventionalT_anno2_UMAP',fig_folder='figures')

### Merge dandelion clone IDs and apply minimum-cell-count donor filter

In [ ]:
total_clone= pd.read_csv('data/99_250611_clone_id_total_ddl.csv', index_col=0)

In [ ]:
total_clone = total_clone[['clone_id']]

In [ ]:
vdata.obs = vdata.obs.merge(total_clone, left_index=True, right_index=True, how='left')

In [ ]:
pco = pd.DataFrame(vdata.obs['PatientID'].value_counts())
ptl = pco[vdata.obs['PatientID'].value_counts()>=314].index.tolist()
print('original {}'.format(len(vdata.obs['PatientID'].unique())))
print('after {}'.format(len(ptl)))


In [ ]:
ptl = pco[pco['count'] >= np.percentile(pco['count'], 5)].index.tolist()


In [ ]:
adata = vdata[vdata.obs['PatientID'].isin(ptl)] 

# Fig. 2B — Distribution of TCR Gini coefficient per T cell subset

In [ ]:
def gini_index(array):
    """
    Compute the Gini coefficient of an array.
    :param array: numeric array representing a distribution
    :return: Gini coefficient (between 0 and 1)
    """
    # sort the array
    array = np.sort(array)
    index = np.arange(1, len(array) + 1)
    n = len(array)
    
    ## compute Gini coefficient
    return ((2 * np.sum(index * array)) / (n * np.sum(array))) - ((n + 1) / n)

In [ ]:
adata.obs['Pt_anno2'] = adata.obs['Pt_anno2'].astype('str')
adata.obs['clone_id_Pt'] = adata.obs['clone_id_Pt'].astype('str')

gindex = {}
for a in adata.obs['Pt_anno2'].unique():
    gindex[a] = gini_index(adata.obs[adata.obs['Pt_anno2']==a]['clone_id_Pt'].value_counts().values)

gdf = pd.DataFrame.from_dict(gindex, orient='index')
gdf = gdf.reset_index()
gdf.columns = ['Pt_anno2','gini_anno2']

test = adata.obs.merge(gdf, on='Pt_anno2', how='left')
test.index = adata.obs.index.copy()

adata.obs = test.copy()

In [ ]:
adata.obs['Pt_anno1'] = adata.obs['Pt_anno1'].astype('str')
adata.obs['clone_id_Pt'] = adata.obs['clone_id_Pt'].astype('str')

gindex = {}
for a in adata.obs['Pt_anno1'].unique():
    gindex[a] = gini_index(adata.obs[adata.obs['Pt_anno1']==a]['clone_id_Pt'].value_counts().values)

gdf = pd.DataFrame.from_dict(gindex, orient='index')
gdf = gdf.reset_index()
gdf.columns = ['Pt_anno1','gini_anno1']

test = adata.obs.merge(gdf, on='Pt_anno1', how='left')
test.index = adata.obs.index.copy()

adata.obs = test.copy()

In [ ]:
adata.obs['Pt_anno0'] = adata.obs['Pt_anno0'].astype('str')
adata.obs['clone_id_Pt'] = adata.obs['clone_id_Pt'].astype('str')

gindex = {}
for a in adata.obs['Pt_anno0'].unique():
    gindex[a] = gini_index(adata.obs[adata.obs['Pt_anno0']==a]['clone_id_Pt'].value_counts().values)

gdf = pd.DataFrame.from_dict(gindex, orient='index')
gdf = gdf.reset_index()
gdf.columns = ['Pt_anno0','gini_anno0']

test = adata.obs.merge(gdf, on='Pt_anno0', how='left')
test.index = adata.obs.index.copy()

adata.obs = test.copy()

In [ ]:
scjp.us(adata, 'gini_anno2,gini_anno1,gini_anno0')

### UMAP overlay of per-cell Gini values (used in Fig. 2 supplementary panels)

In [ ]:
scjp.us(adata, 'gini_anno2', frameon=False, s=0.5, vmax=0.9)
plt.title('')
scjp.save_fig('Fig2_','T_GINI_UMAP',fig_folder='figures')

### Boxplot of Gini per T cell subset (Fig. 2B)

In [ ]:
g2 = adata.obs.groupby('Pt_anno2').mean()[['gini_anno2']]

g2['PatientID'] = [a.split('*')[0] for a in g2.index]
g2['anno2'] = [a.split('*')[1] for a in g2.index]

g2 = g2.merge(meta.reset_index(), on='PatientID', how='left')

In [ ]:
poco = adata.obs['Pt_anno2'].value_counts()[adata.obs['Pt_anno2'].value_counts() >= 10].index
g2 = adata.obs.groupby('Pt_anno2').mean()[['gini_anno2']]
g2['PatientID'] = [a.split('*')[0] for a in g2.index]
g2['anno2'] = [a.split('*')[1] for a in g2.index]
g2 = g2[g2.index.isin(poco)]
g2 = g2.merge(meta.reset_index(), on='PatientID', how='left')

plt.figure(figsize=(5, 6))
order = [c for c in T_anno2_order if c in g2['anno2'].unique()]
sns.boxplot(
    data=g2, x='anno2', y='gini_anno2',
    order=order, palette=T_anno2_colors, showfliers=False
)
sns.stripplot(data=g2, x='anno2', y='gini_anno2', order=order, color='k', size=1, alpha=0.4)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Gini coefficient')
plt.xlabel('')
sns.despine()
plt.tight_layout()
plt.savefig('figures/Fig2_gini_anno2_barplot.pdf', dpi=300, format='pdf', transparent=True, bbox_inches='tight')
plt.show()


# Fig. 2C — Rarefied Gini coefficient vs. age, per T cell subset

To control for the strong dependence of Gini on per-donor sample size, we subsample each
donor x cell type to a common cell depth (`raref_depth_map`) and average Gini across
`n_repeat_raref` rarefied draws. The depth is chosen separately for each subset to
maximise donor inclusion.

In [ ]:

# -----------------------------
# settings
# -----------------------------
clone_col = "clone_id_Pt"   # <- modify if needed
ptanno_col = "Pt_anno2"

raref_depth_map = {
    "T_CD8_TEM_GZMK": 50,
    "T_CD8_TEM_GZMB": 50,
    "T_CD8_KIR":30,
    "T_CD4_CTL": 30,
}

target_cells = list(raref_depth_map.keys())
n_repeat_raref = 100

# -----------------------------
# helper
# -----------------------------
def gini_from_counts(counts):
    x = np.asarray(counts, dtype=float)
    if len(x) == 0 or np.sum(x) == 0:
        return np.nan
    x = np.sort(x)
    n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x)) / (n * np.sum(x)) - (n + 1) / n

def rarefied_gini(clone_ids, depth, n_repeat=200, random_state=0):
    clone_ids = np.asarray(clone_ids)
    n_total = len(clone_ids)
    if n_total < depth:
        return np.nan

    rng = np.random.default_rng(random_state)
    vals = []
    for i in range(n_repeat):
        idx = rng.choice(n_total, size=depth, replace=False)
        sampled = clone_ids[idx]
        counts = pd.Series(sampled).value_counts().values
        vals.append(gini_from_counts(counts))
    return np.mean(vals)


In [ ]:

# -----------------------------
# 1) keep only usable cells
# -----------------------------
obs = adata.obs.copy()

tmp = obs[[ptanno_col, clone_col]].copy()
tmp = tmp.dropna(subset=[ptanno_col, clone_col]).copy()

tmp[ptanno_col] = tmp[ptanno_col].astype(str)
tmp[clone_col] = tmp[clone_col].astype(str)

tmp = tmp[
    (~tmp[clone_col].isin(["", "nan", "None", "NA"])) &
    (tmp[clone_col].str.strip() != "")
].copy()

tmp["PatientID"] = tmp[ptanno_col].str.split("*").str[0]
tmp["anno2"] = tmp[ptanno_col].str.split("*").str[1]

tmp = tmp[tmp["anno2"].isin(target_cells)].copy()

# -----------------------------
# 2) compute rarefied Gini per Pt_anno2
# -----------------------------
rows = []

for ptanno, sub in tmp.groupby(ptanno_col):
    cell = sub["anno2"].iloc[0]
    depth = raref_depth_map[cell]
    n_cells = len(sub)

    if n_cells < depth:
        continue

    rg = rarefied_gini(
        clone_ids=sub[clone_col].values,
        depth=depth,
        n_repeat=n_repeat_raref,
        random_state=0
    )

    rows.append({
        "Pt_anno2": ptanno,
        "PatientID": sub["PatientID"].iloc[0],
        "anno2": cell,
        "n_cells_anno2": n_cells,
        "raref_depth": depth,
        "gini_rarefied": rg
    })

g2r = pd.DataFrame(rows)

# -----------------------------
# 3) meta merge
# -----------------------------
g2r = g2r.merge(meta.reset_index(), on="PatientID", how="left")


In [ ]:
# -----------------------------
# 4) plot
# -----------------------------
for cell in target_cells:
    ddf = g2r[g2r["anno0"] == cell].copy()
    ddf = ddf.dropna(subset=["Age", "Sex", "gini_rarefied"])

    if ddf.shape[0] < 3:
        print(f"skip {cell}: too few samples")
        continue

    g = sns.lmplot(
        data=ddf,
        x="Age",
        y="gini_rarefied",
        hue="Sex",
        hue_order=["Male", "Female"],
        palette=sex_colors,
        scatter_kws={"ec": "black", "s": 30, "alpha": 0.7},
        line_kws={"lw": 3, "ls": "--", "alpha": 0.8},
        height=5,
        aspect=1
    )

    # line outline
    for ax in g.axes.flat:
        for line in ax.lines:
            line.set_path_effects([
                path_effects.Stroke(linewidth=5, foreground="black"),
                path_effects.Normal()
            ])

    # overall Pearson
    pv = stats.pearsonr(ddf["Age"], ddf["gini_rarefied"])

    for ax in g.axes.flat:
        ax.set_title(f"{cell} (depth={raref_depth_map[cell]})", color="k", fontsize=20)
        ax.set_xlabel(f"Age\nr={pv[0]:.3f}, p={pv[1]:.3f}", fontsize=16)
        ax.set_ylabel("Rarefied Gini coefficient", fontsize=16)
        sns.despine(ax=ax)
    plt.savefig('figures/Fig2_rare_gini_{}_lmplot.pdf'.format(cell), dpi=300, format='pdf',transparent=True, bbox_inches='tight')
    plt.show()

### Rarefied Gini vs. age — global CD4+ and CD8+ pools

In [ ]:

# -----------------------------
# settings
# -----------------------------
clone_col = "clone_id_Pt"   # <- modify if needed
ptanno_col = "Pt_anno0"

raref_depth_map = {
    "T_CD8": 200,
    "T_CD4": 200,
}

target_cells = list(raref_depth_map.keys())
n_repeat_raref = 100

# -----------------------------
# helper
# -----------------------------
def gini_from_counts(counts):
    x = np.asarray(counts, dtype=float)
    if len(x) == 0 or np.sum(x) == 0:
        return np.nan
    x = np.sort(x)
    n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x)) / (n * np.sum(x)) - (n + 1) / n

def rarefied_gini(clone_ids, depth, n_repeat=200, random_state=0):
    clone_ids = np.asarray(clone_ids)
    n_total = len(clone_ids)
    if n_total < depth:
        return np.nan

    rng = np.random.default_rng(random_state)
    vals = []
    for i in range(n_repeat):
        idx = rng.choice(n_total, size=depth, replace=False)
        sampled = clone_ids[idx]
        counts = pd.Series(sampled).value_counts().values
        vals.append(gini_from_counts(counts))
    return np.mean(vals)


In [ ]:

# -----------------------------
# 1) keep only usable cells
# -----------------------------
obs = adata.obs.copy()

tmp = obs[[ptanno_col, clone_col]].copy()
tmp = tmp.dropna(subset=[ptanno_col, clone_col]).copy()

tmp[ptanno_col] = tmp[ptanno_col].astype(str)
tmp[clone_col] = tmp[clone_col].astype(str)

tmp = tmp[
    (~tmp[clone_col].isin(["", "nan", "None", "NA"])) &
    (tmp[clone_col].str.strip() != "")
].copy()

tmp["PatientID"] = tmp[ptanno_col].str.split("*").str[0]
tmp["anno0"] = tmp[ptanno_col].str.split("*").str[1]

tmp = tmp[tmp["anno0"].isin(target_cells)].copy()

# -----------------------------
# 2) compute rarefied Gini per Pt_anno0
# -----------------------------
rows = []

for ptanno, sub in tmp.groupby(ptanno_col):
    cell = sub["anno0"].iloc[0]
    depth = raref_depth_map[cell]
    n_cells = len(sub)

    if n_cells < depth:
        continue

    rg = rarefied_gini(
        clone_ids=sub[clone_col].values,
        depth=depth,
        n_repeat=n_repeat_raref,
        random_state=0
    )

    rows.append({
        "Pt_anno0": ptanno,
        "PatientID": sub["PatientID"].iloc[0],
        "anno0": cell,
        "n_cells_anno0": n_cells,
        "raref_depth": depth,
        "gini_rarefied": rg
    })

g2r = pd.DataFrame(rows)

# -----------------------------
# 3) meta merge
# -----------------------------
g2r = g2r.merge(meta.reset_index(), on="PatientID", how="left")


In [ ]:
# -----------------------------
# 4) plot
# -----------------------------
for cell in target_cells:
    ddf = g2r[g2r["anno0"] == cell].copy()
    ddf = ddf.dropna(subset=["Age", "Sex", "gini_rarefied"])

    if ddf.shape[0] < 3:
        print(f"skip {cell}: too few samples")
        continue

    g = sns.lmplot(
        data=ddf,
        x="Age",
        y="gini_rarefied",
        hue="Sex",
        hue_order=["Male", "Female"],
        palette=sex_colors,
        scatter_kws={"ec": "black", "s": 30, "alpha": 0.7},
        line_kws={"lw": 3, "ls": "--", "alpha": 0.8},
        height=5,
        aspect=1
    )

    # line outline
    for ax in g.axes.flat:
        for line in ax.lines:
            line.set_path_effects([
                path_effects.Stroke(linewidth=5, foreground="black"),
                path_effects.Normal()
            ])

    # overall Pearson
    pv = stats.pearsonr(ddf["Age"], ddf["gini_rarefied"])

    for ax in g.axes.flat:
        ax.set_title(f"{cell} (depth={raref_depth_map[cell]})", color="k", fontsize=20)
        ax.set_xlabel(f"Age\nr={pv[0]:.3f}, p={pv[1]:.3f}", fontsize=16)
        ax.set_ylabel("Rarefied Gini coefficient", fontsize=16)
        sns.despine(ax=ax)
    plt.savefig('figures/Fig2_rare_gini_{}_lmplot.pdf'.format(cell), dpi=300, format='pdf',transparent=True, bbox_inches='tight')
    plt.show()

# Fig. 2D, 2E — Public clones across ethnicities

### Load TCR contigs (`dandelion`) and call public clones

Public clones are defined as clone IDs (Dandelion identity = 1.0, paired αβ) that are
shared across two or more donors.

In [ ]:
vdata = sc.read('data/06_250601_TDATA_dandelion_TCR.h5ad')

In [ ]:
tcr = ddl.read_10x_airr('data/240716_TCR_ALL_AIRR.tsv')

In [ ]:
vdj, vdata = ddl.pp.filter_contigs(tcr, vdata, library_type='tr-ab') 

In [ ]:
vdata = vdata[~vdata.obs.anno2.str.startswith('NK')]
vdata = vdata[~vdata.obs.anno2.str.startswith('T_unc_gd')]

In [ ]:
ddl.tl.find_clones(vdj, identity=1.0)

In [ ]:
ddl.tl.clone_size(vdj)

In [ ]:
ddl.tl.transfer(vdata, vdj)

In [ ]:
vdata = vdata[vdata.obs['Ethnicity']!='European']
vdata = vdata[~vdata.obs['Ethnicity'].isna()]

Per-donor metadata frame and harmonised category order:

In [ ]:
meta = vdata.obs.drop_duplicates('PatientID').set_index('PatientID')[['Age','Sex','Ethnicity']]

### Identify public clones (≥2 donors carrying the same clone)

In [ ]:
cf= pd.crosstab(vdata.obs['clone_id'],vdata.obs['PatientID'])

non_zero_counts = (cf != 0).sum(axis=1)

result = cf[non_zero_counts > 1]

result = result.T

result=result.reset_index().set_index('PatientID')

pclone = result.columns

vdata.obs['Public'] = 'No'
vdata.obs['Public'] = np.where(vdata.obs['clone_id'].isin(pclone), 'Public', vdata.obs['Public'])

In [ ]:
vdata.obs['anno2'] = vdata.obs['anno2'].cat.reorder_categories([   'T_CD4_Naive_SOX4',  'T_CD4_Naive','T_CD4_cTfh','T_CD4_Th1', 'T_CD4_Th2',  'T_CD4_Th17','T_CD4_activated', 
               'T_CD4_CTL',       'T_CD4_Treg', 'T_CD4_IFN',
               'T_CD8_Naive_SOX4', 'T_CD8_Naive', 'T_CD8_TEM_GZMK', 'T_CD8_TEM_GZMB','T_CD8_KIR',
       'T_unc_MAIT','T_unc_dnT'])

### Stacked bar plot of public proportion across cell subsets

In [ ]:
dpf = pd.crosstab(vdata.obs['anno2'], vdata.obs['Public'], normalize=0)
dpf = dpf*100

dpf = dpf[['Public','No']]

ax = dpf.plot(kind='barh', stacked=True,                 
         width=0.95, ec='k', lw=0.5, color=['orangered','lightgray',], alpha=0.7)
# plt.ylim([0,0.3])
plt.legend(loc=(1.02,0))
plt.xlabel('Proportion %')
plt.ylabel('')
plt.xlim([0,13])
sns.despine()
# ax.get_legend().remove()
plt.title('Public clone proportion across the cell type')

### Fig. 2D — UMAP overlay of public clone locations

In [ ]:
plt.figure(figsize=(7,7))
sns.scatterplot(x=vdata.obsm['X_umap'][:,0], y=vdata.obsm['X_umap'][:,1], s=1, color='silver')
sns.scatterplot(x=vdata[vdata.obs['Public']=='Public'].obsm['X_umap'][:,0],y=vdata[vdata.obs['Public']=='Public'].obsm['X_umap'][:,1], 
                s=2, ec='k', color='coral')
# plt.title('T cell with CD8+GZMK+TEM public clone CDR3b')
sns.despine()

### Count number of ethnicities per public clone (`Eth_num`)

In [ ]:
sdata = vdata[vdata.obs['Public']=='Public']

In [ ]:
publics = sdata.obs.copy()

In [ ]:
pnum = {}
for a in publics['clone_id'].unique():
    # collect all unique Ethnicity values
    all_ethnicities = publics['Ethnicity'].unique()

    # store value_counts() result as Series
    counts = publics[publics.clone_id == a]['Ethnicity'].value_counts()

    # build Series including 0s for all Ethnicities
    counts_with_zeros = counts.reindex(all_ethnicities, fill_value=0)

    # count zero entries
    pnum[a] =  6- (counts_with_zeros == 0).sum()

pnumf = pd.DataFrame.from_dict(pnum,orient='index',columns=['Eth_num'])

In [ ]:
pn = pnumf.reset_index() 

pn['clone_id']= pn['index'].copy()
del pn['index']

brid = sdata.obs.merge(pn, on='clone_id', how='left')

brid.index = sdata.obs.index.copy()

sdata.obs= brid.copy()


In [ ]:
sdf = (
    sdata.obs
      .assign(
         UMAP1 = sdata.obsm['X_umap'][:,0],
         UMAP2 = sdata.obsm['X_umap'][:,1]
      )
)


In [ ]:
sdf = sdf.sort_values('Eth_num')

In [ ]:
plt.figure(figsize=(7,7))
sns.scatterplot(x=vdata.obsm['X_umap'][:,0], y=vdata.obsm['X_umap'][:,1], s=1, color='silver', alpha=0.7, 
            rasterized=True)
sns.scatterplot(
    data=sdf,
    x='UMAP1', y='UMAP2',   
    hue='Eth_num',
         # or Reds, or whatever
    size='Eth_num',          # map the variable to size
    sizes=(2, 10),         # smallest point 10², largest 100² (you can tweak)
    edgecolor='k',
    legend='brief', alpha=0.8,
     rasterized=True)
plt.title("UMAP, point size ∝ Eth_num")
sns.despine()
plt.savefig('figures/Fig2_public_scatter_umap_identical.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')

plt.show()

 

### Fig. 2E — Public clone proportion per cell type, stratified by sharing degree

In [ ]:
vvdata = vdata.copy()

In [ ]:
brid = vvdata.obs.merge(pn, on='clone_id', how='left')

brid.index = vvdata.obs.index.copy()

vvdata.obs= brid.copy()


In [ ]:
vvdata.obs['Eth_num'] = np.where(vvdata.obs['Eth_num'].isna(), 0, vvdata.obs['Eth_num'])

In [ ]:
colors = sns.cubehelix_palette(n_colors=6)
colors.append('lightgray')


In [ ]:
vvdata.obs['anno2']  = vvdata.obs['anno2'].astype('category')

In [ ]:
vvdata.obs['anno2'] = vvdata.obs['anno2'].cat.reorder_categories([   'T_CD4_Naive_SOX4',  'T_CD4_Naive','T_CD4_cTfh','T_CD4_Th1', 'T_CD4_Th2',  'T_CD4_Th17','T_CD4_activated', 
               'T_CD4_CTL',       'T_CD4_Treg', 'T_CD4_IFN',
               'T_CD8_Naive_SOX4', 'T_CD8_Naive', 'T_CD8_TEM_GZMK', 'T_CD8_TEM_GZMB','T_CD8_KIR',
       'T_unc_MAIT','T_unc_dnT'])

In [ ]:
dpf = pd.crosstab(vvdata.obs['anno2'], vvdata.obs['Eth_num'], normalize=0)
dpf = dpf*100
dpf= dpf[[1,2,3,4,5,6,0]]
dpf = dpf.T
dpf = dpf[dpf.columns[::-1]].T
ax = dpf.plot(kind='barh', stacked=True,                 
         width=0.95, ec='k', lw=0.5, alpha=0.8,
              color=colors
             )
# plt.ylim([0,0.3])
plt.legend(loc=(1.02,0))
plt.ylabel('')
plt.xlabel('proportion %')
plt.xlim([0,8.1])
sns.despine()
# ax.get_legend().remove()
plt.title('Public clone proportion across the cell type')

plt.savefig('figures/Fig2_public_per_celltype_barplot_identical.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')


# Fig. 2F — VDJdb antigen prediction by sharing breadth

VDJdb is queried with `scirpy.tl.ir_query` using identity matching on the amino-acid
junction sequence. We then count the fraction of clonotypes annotated with a known
epitope specificity per sharing degree (Eth_num).

In [ ]:
import scirpy as ir

In [ ]:
from tqdm import tqdm

vdjdbdf = pd.read_csv("reference/vdjdb/vdjdb_full.txt", sep="\t")

In [ ]:
# VDJdb reference TSV (https://vdjdb.cdr3.net)
vdjdbdf = pd.read_csv("reference/vdjdb/vdjdb_full.txt", sep="\t")


In [ ]:
tcr_cells = []
for idx, row in tqdm(
    vdjdbdf.iterrows(), total=vdjdbdf.shape[0], desc="Processing VDJDB entries"
):
    cell = ir.io.AirrCell(cell_id=idx)
    if not pd.isnull(row["cdr3.alpha"]):
        alpha_chain = ir.io.AirrCell.empty_chain_dict()
        alpha_chain.update(
            {
                "locus": "TRA",
                "junction_aa": row["cdr3.alpha"],
                "v_call": row["v.alpha"],
                "j_call": row["j.alpha"],
                "consensus_count": 0,
                "productive": True,
            }
        )
        cell.add_chain(alpha_chain)

    if not pd.isnull(row["cdr3.beta"]):
        beta_chain = ir.io.AirrCell.empty_chain_dict()
        beta_chain.update(
            {
                "locus": "TRB",
                "junction_aa": row["cdr3.beta"],
                "v_call": row["v.beta"],
                "d_call": row["d.beta"],
                "j_call": row["j.beta"],
                "consensus_count": 0,
                "productive": True,
            }
        )
        cell.add_chain(beta_chain)

    INCLUDE_CELL_METADATAFIELDS = [
        "species",
        "mhc.a",
        "mhc.b",
        "mhc.class",
        "antigen.epitope",
        "antigen.gene",
        "antigen.species",
        "reference.id",
        "method.identification",
        "method.frequency",
        "method.singlecell",
        "method.sequencing",
        "method.verification",
        "meta.study.id",
        "meta.cell.subset",
        "meta.subject.cohort",
        "meta.subject.id",
        "meta.replica.id",
        "meta.clone.id",
        "meta.epitope.id",
        "meta.tissue",
        "meta.donor.MHC",
        "meta.donor.MHC.method",
        "meta.structure.id",
    ]
    for f in INCLUDE_CELL_METADATAFIELDS:
        cell[f] = row[f]
    tcr_cells.append(cell)
from datetime import datetime
from scanpy import logging
logging.info("Converting to AnnData object")

In [ ]:
vdjdata = ir.io.from_airr_cells(tcr_cells)

vdjdata.uns["DB"] = {"name": "VDJDB", "date_downloaded": datetime.now().isoformat()}

Use the Pt_anno1 / Pt_anno2 reference object for the query (which contains the standard scirpy fields).

In [ ]:
fdata = sc.read('data/06_250601_TDATA_scirpy_merging_meta_with_clone_id_Pt.h5ad')

In [ ]:
fdata = fdata[fdata.obs['Ethnicity']!='European']
fdata = fdata[~fdata.obs['Ethnicity'].isna()]

In [ ]:
ir.pp.ir_dist(fdata, vdjdata, metric="identity", sequence="aa")

In [ ]:
ir.tl.ir_query(
    fdata, vdjdata, metric="identity", sequence="aa", receptor_arms="any", dual_ir="any"
)

vq = ir.tl.ir_query_annotate_df(
    fdata,
    vdjdata,
    metric="identity",
    sequence="aa",
    include_ref_cols=["antigen.species", "antigen.gene"],
)

ir.tl.ir_query_annotate(
    fdata,
    vdjdata,
    metric="identity",
    sequence="aa",
    include_ref_cols=["antigen.species"], 
)

### Build VDJdb call status table per donor

In [ ]:
fdf = fdata.obs[['Ethnicity','PatientID','anno2','antigen.species']]

In [ ]:
fdf['antigen.species'] = fdf['antigen.species'].astype('str')

In [ ]:
fdf['vdjdb_calling'] = 'Predicted'

In [ ]:
fdf['vdjdb_calling'] = np.where(fdf['antigen.species']=='nan', 'NaN', fdf['vdjdb_calling'])
fdf['vdjdb_calling'] = np.where(fdf['antigen.species']=='ambiguous', 'ambiguous', fdf['vdjdb_calling'])

In [ ]:
fdf['vdjdb_calling'].value_counts()/len(fdf['vdjdb_calling'])

In [ ]:
pdf = pd.crosstab(fdf['Ethnicity'], fdf['vdjdb_calling'], normalize=0)[['NaN','ambiguous','Predicted'][::-1]]

In [ ]:
pdf.plot(kind='barh', stacked=True, width=0.9, figsize=(6,3),
        ec='k')
sns.despine()
# plt.savefig('figures/supple/VDJDB_predicted.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')


### Fig. 2F — Fraction of clonotypes annotated to specific antigens by sharing breadth

We stratify clones by `Eth_num` (number of ethnicities sharing the clone) and compute
the proportion of clones with a VDJdb-predicted antigen specificity per group.
Predicted pathogens of interest: CMV, EBV, Influenza A, SARS-CoV-2.

In [ ]:
# Build per-clone Eth_num + antigen.species table
clone_eth = vvdata.obs[['clone_id', 'Eth_num', 'antigen.species']].drop_duplicates('clone_id').dropna(subset=['clone_id'])
clone_eth['antigen.species'] = clone_eth['antigen.species'].astype(str)

# Fraction predicted per sharing degree
prop = pd.crosstab(clone_eth['Eth_num'], clone_eth['antigen.species'], normalize=0) * 100
# Drop unannotated columns
keep_cols = [c for c in prop.columns if c not in ['nan', 'NaN', 'None', '']]
prop = prop[keep_cols]
prop = prop.loc[:, prop.mean() > 0.1]  # drop very rare antigens

prop.plot(kind='bar', stacked=True, figsize=(6, 4), width=0.85, ec='k', lw=0.5)
plt.legend(loc=(1.02, 0))
plt.ylabel('Proportion of predicted clones (%)')
plt.xlabel('Number of ethnicities sharing the clone')
sns.despine()
plt.title('VDJdb-predicted antigen specificity by sharing breadth')
plt.tight_layout()
plt.savefig('figures/Fig2_VDJdb_predicted_by_Ethnum.pdf', dpi=300, format='pdf', transparent=True, bbox_inches='tight')
plt.show()


# Fig. 2G — Representative public clones (junction logos and clone passports)

Helper functions to render two-panel TRA/TRB sequence logos and the per-clone "passport"
showing clone size, donor / ethnicity sharing, dominant cell type, predicted antigen,
HLA usage, and CDR3 motif.

In [ ]:
import matplotlib as mpl

mpl.rcParams['font.family'] = 'Liberation Sans'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

In [ ]:
kr = pd.read_csv('../../HLA_arcas/20230315_KR_arcasHLA_lane_merged_result.genotypes.resolution_lv_p_group.modified.tsv', sep='\t')

df_melt = kr.melt(
    id_vars=["subject"], 
    var_name="Locus", 
    value_name="Allele"
)

df_melt["Allele"] = [a.split(':')[0] for a in df_melt['Allele']]

df_melt["present"] = 1

df_melt = df_melt.drop_duplicates(["subject", "Allele"])

kr_df = df_melt.pivot_table(
    index="subject",
    columns="Allele",
    values="present",
    fill_value=0
).reset_index()

jp = pd.read_csv('../../HLA_arcas/20230315_JP_arcasHLA_lane_merged_result.genotypes.resolution_lv_p_group.modified.tsv', sep='\t')

df_melt = jp.melt(
    id_vars=["subject"], 
    var_name="Locus", 
    value_name="Allele"
)

df_melt["Allele"] = [a.split(':')[0] for a in df_melt['Allele']]

df_melt["present"] = 1

df_melt = df_melt.drop_duplicates(["subject", "Allele"])

jp_df = df_melt.pivot_table(
    index="subject",
    columns="Allele",
    values="present",
    fill_value=0
).reset_index()

sg = pd.read_csv('../../HLA_arcas/20230315_SG_Chinese_arcasHLA_lane_merged_result.genotypes.resolution_lv_p_group.modified.tsv', sep='\t')

df_melt = sg.melt(
    id_vars=["subject"], 
    var_name="Locus", 
    value_name="Allele"
)

df_melt["Allele"] = [a.split(':')[0] for a in df_melt['Allele']]

df_melt["present"] = 1

df_melt = df_melt.drop_duplicates(["subject", "Allele"])

sgchi_df = df_melt.pivot_table(
    index="subject",
    columns="Allele",
    values="present",
    fill_value=0
).reset_index()

sg = pd.read_csv('../../HLA_arcas/20230315_SG_Malay_arcasHLA_lane_merged_result.genotypes.resolution_lv_p_group.modified.tsv', sep='\t')

df_melt = sg.melt(
    id_vars=["subject"], 
    var_name="Locus", 
    value_name="Allele"
)

df_melt["Allele"] = [a.split(':')[0] for a in df_melt['Allele']]

df_melt["present"] = 1

df_melt = df_melt.drop_duplicates(["subject", "Allele"])

sgmal_df = df_melt.pivot_table(
    index="subject",
    columns="Allele",
    values="present",
    fill_value=0
).reset_index()

sg = pd.read_csv('../../HLA_arcas/20230315_SG_Indian_arcasHLA_lane_merged_result.genotypes.resolution_lv_p_group.modified.tsv', sep='\t')

df_melt = sg.melt(
    id_vars=["subject"], 
    var_name="Locus", 
    value_name="Allele"
)

df_melt["Allele"] = [a.split(':')[0] for a in df_melt['Allele']]

df_melt["present"] = 1

df_melt = df_melt.drop_duplicates(["subject", "Allele"])

sgind_df = df_melt.pivot_table(
    index="subject",
    columns="Allele",
    values="present",
    fill_value=0
).reset_index()

hladf = pd.concat([kr_df,jp_df,sgchi_df,sgind_df,sgmal_df])
hladf = hladf.fillna(0)

In [ ]:
import ast
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, PathPatch
from matplotlib.path import Path
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
def plot_clone_passport(
    clone,
    vvdata,
    hladf,
    Ethnicity_order=('Chinese', 'Malay', 'Indian', 'Japanese', 'Korean', 'Thai'),
    Ethnicity_colors=("tomato", "gold", "dodgerblue", "limegreen", "aquamarine", "orchid"),
    figsize=(18, 7),
    type='CD8',
    dpi=300,
    save_path=None,
    show=True
):
    eth_color_dict = dict(zip(Ethnicity_order, Ethnicity_colors))

    # -----------------------------
    # helpers
    # -----------------------------
    def pick_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    def representative_value(df, col):
        if col is None:
            return None
        s = df[col].dropna().astype(str)
        if len(s) == 0:
            return None
        return s.value_counts().index[0]

    def parse_listlike_field(x):
        if pd.isna(x):
            return []
        if isinstance(x, (list, tuple, set, np.ndarray)):
            vals = list(x)
        else:
            s = str(x).strip()
            if s in ['', 'nan', 'None', 'NA', 'Unknown', 'unknown']:
                return []
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, (list, tuple, set, np.ndarray)):
                    vals = list(obj)
                else:
                    vals = [s]
            except Exception:
                vals = re.split(r'[;,|/]+', s)

        vals = [str(v).strip() for v in vals]
        vals = [v for v in vals if v not in ['', 'nan', 'None', 'NA', 'Unknown', 'unknown']]
        return vals

    def truncate_text(s, max_len=34):
        s = str(s)
        return s if len(s) <= max_len else s[:max_len-1] + '…'

    def compute_block_positions(labels, totals, y_top=0.72, y_bottom=0.10, gap=0.02):
        labels = list(labels)
        n = len(labels)
        if n == 0:
            return {}
        total = float(sum(totals[l] for l in labels))
        available = y_top - y_bottom - gap * (n - 1)
        y = y_top
        pos = {}
        for lab in labels:
            h = available * (totals[lab] / total) if total > 0 else available / n
            pos[lab] = (y - h, y)
            y = y - h - gap
        return pos

    def ribbon_patch(x0, x1, y0a, y0b, y1a, y1b, color, alpha=0.28):
        c = 0.35 * (x1 - x0)
        verts = [
            (x0, y0b),
            (x0 + c, y0b),
            (x1 - c, y1b),
            (x1, y1b),

            (x1, y1a),
            (x1 - c, y1a),
            (x0 + c, y0a),
            (x0, y0a),
            (x0, y0b)
        ]
        codes = [
            Path.MOVETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,

            Path.LINETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,
            Path.CLOSEPOLY
        ]
        return PathPatch(Path(verts, codes), facecolor=color, edgecolor='none', alpha=alpha)

    def draw_summary_bar(ax, text, x=0.03, y=0.57, w=0.91, h=0.09):
        patch = FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.012,rounding_size=0.012",
            linewidth=0.8, edgecolor='#b7b7b7', facecolor='#f5f5f5'
        )
        ax.add_patch(patch)
        ax.text(x + 0.01, y + h/2, text, fontsize=11, va='center', ha='left', color='#222222')
    anno2_order = [
        'T_CD8_Naive',
        'T_CD4_Naive',
        'T_CD8_TEM_GZMB',
        'T_CD4_Th2',
        'T_CD4_Th1',
        'T_CD8_TEM_GZMK',
        'T_CD4_CTL',
        'T_CD4_cTfh',
        'T_CD4_Naive_SOX4',
        'T_CD4_Th17',
        'T_unc_MAIT',
        'T_CD4_activated',
        'T_CD4_Treg',
        'T_CD8_KIR',
        'T_CD8_Naive_SOX4',
        'T_CD4_IFN',
        'T_unc_dnT'
    ]

    anno2_colors = [
        '#d6bcc0', '#bb7784', '#8e063b', '#4a6fe3', '#8595e1', '#b5bbe3',
        '#e6afb9', '#e07b91', '#d33f6a', '#11c638', '#8dd593', '#c6dec7',
        '#ead3c6', '#f0b98d', '#ef9708', '#0fcfc0', '#9cded6'
    ]
    anno2_color_dict = dict(zip(anno2_order, anno2_colors))
    # -----------------------------
    # 1) clone subset
    # -----------------------------
    obs = vvdata.obs.copy()
    clone_obs = obs[obs['clone_id'] == clone].copy()
    if clone_obs.shape[0] == 0:
        raise ValueError(f'Clone not found: {clone}')

    # -----------------------------
    # 2) sequence columns
    # -----------------------------
    cdr3a_col = pick_col(clone_obs, ['junction_aa_VJ', 'cdr3a', 'CDR3a', 'TRA_cdr3'])
    cdr3b_col = pick_col(clone_obs, ['junction_aa_VDJ', 'cdr3b', 'CDR3b', 'TRB_cdr3'])

    trav_col = pick_col(clone_obs, ['IR_VJ_1_v_call', 'v_call_VJ_main', 'TRAV'])
    traj_col = pick_col(clone_obs, ['IR_VJ_1_j_call', 'j_call_VJ_main', 'TRAJ'])
    trbv_col = pick_col(clone_obs, ['IR_VDJ_1_v_call', 'v_call_VDJ_main', 'TRBV'])
    trbj_col = pick_col(clone_obs, ['IR_VDJ_1_j_call', 'j_call_VDJ_main', 'TRBJ'])

    cdr3a = representative_value(clone_obs, cdr3a_col)
    cdr3b = representative_value(clone_obs, cdr3b_col)
    trav  = representative_value(clone_obs, trav_col)
    traj  = representative_value(clone_obs, traj_col)
    trbv  = representative_value(clone_obs, trbv_col)
    trbj  = representative_value(clone_obs, trbj_col)

    # -----------------------------
    # 3) donor + HLA
    # -----------------------------
    donors = clone_obs[['DCP_ID', 'Ethnicity']].drop_duplicates().copy()
    donor_origin = donors.copy()
    donors['DCP_ID'] = donors['DCP_ID'].astype(str)
    donors = donors[donors['DCP_ID'] != 'nan'].copy()

    hla = hladf.copy()
    hla['subject'] = hla['subject'].astype(str)
    donor_hla = donors.merge(hla, left_on='DCP_ID', right_on='subject', how='left')

    if type == 'CD8':
        allele_cols = [c for c in donor_hla.columns if c.startswith(('A', 'B', 'C'))]
        group_titles = ('HLA-A', 'HLA-B', 'HLA-C')
        prefix_groups = ('A', 'B', 'C')
    else:
        allele_cols = [c for c in donor_hla.columns if c.startswith('D')]
        group_titles = ('HLA-DR', 'HLA-DQ', 'HLA-DP')
        prefix_groups = ('DR', 'DQ', 'DP')

    if len(allele_cols) > 0:
        donor_hla[allele_cols] = (
            donor_hla[allele_cols]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0)
            .astype(int)
        )

        n_donors_hla = max(1, donors['DCP_ID'].nunique())
        hla_freq = donor_hla[allele_cols].sum(axis=0).sort_values(ascending=False)
        major_hla = hla_freq[hla_freq / n_donors_hla >= 0.5]
        if len(major_hla) == 0:
            major_hla = hla_freq.head(6)

        major_hla_A = [x for x in major_hla.index if str(x).startswith(prefix_groups[0])]
        major_hla_B = [x for x in major_hla.index if str(x).startswith(prefix_groups[1])]
        major_hla_C = [x for x in major_hla.index if str(x).startswith(prefix_groups[2])]
        selected_hla = major_hla.index.tolist()
        heatmap_hla = hla_freq[hla_freq > 0].index.tolist()
    else:
        hla_freq = pd.Series(dtype=float)
        major_hla_A, major_hla_B, major_hla_C, selected_hla = [], [], [], []

    # -----------------------------
    # 3b) VDJdb inferred species
    # -----------------------------
    species_col = 'antigen.species' if 'antigen.species' in clone_obs.columns else None
    if species_col is not None:
        species_hits = (
            clone_obs[species_col]
            .dropna()
            .drop_duplicates()
            .apply(parse_listlike_field)
            .explode()
            .dropna()
        )
        species_hits = species_hits[species_hits.astype(str).str.strip() != '']
        species_freq = species_hits.value_counts()
        major_species = species_freq.index.tolist()
    else:
        species_freq = pd.Series(dtype=int)
        major_species = []

    # -----------------------------
    # 4) UMAP dataframe
    # -----------------------------
    tmp = vvdata.obs.copy()
    tmp['highlight'] = 'Other'
    mask = tmp['clone_id'] == clone
    tmp.loc[mask, 'highlight'] = tmp.loc[mask, 'Ethnicity']

    umap = vvdata.obsm['X_umap']
    plot_df = pd.DataFrame({
        'UMAP1': umap[:, 0],
        'UMAP2': umap[:, 1],
        'highlight': tmp['highlight'].values
    })

    bg_df = plot_df.loc[plot_df['highlight'] == 'Other']


    # -----------------------------
    # 5b) cell type proportion for pie
    # -----------------------------
    celltype_counts = (
        clone_obs['anno2']
        .fillna('NA')
        .astype(str)
        .value_counts()
    )

    # sort in the specified order
    celltype_counts = celltype_counts.reindex(
        [x for x in anno2_order if x in celltype_counts.index]
    ).dropna()

    # append items missing from anno2_order at the end
    extra_ct = [
        x for x in clone_obs['anno2'].fillna('NA').astype(str).unique()
        if x not in anno2_order
    ]
    if len(extra_ct) > 0:
        extra_counts = (
            clone_obs['anno2']
            .fillna('NA')
            .astype(str)
            .value_counts()
            .reindex(extra_ct)
            .dropna()
        )
        celltype_counts = pd.concat([celltype_counts, extra_counts])

    top_n_ct = 5
    if len(celltype_counts) > top_n_ct:
        celltype_plot = pd.concat([
            celltype_counts.iloc[:top_n_ct],
            pd.Series({'Other': celltype_counts.iloc[top_n_ct:].sum()})
        ])
    else:
        celltype_plot = celltype_counts.copy()
    
    # -----------------------------
    # 5b) donor x HLA heatmap matrix
    # -----------------------------
    hm_eth_order = [e for e in Ethnicity_order if e != 'Thai']   # exclude Thai

    # fallback if selected_hla is empty
    if len(heatmap_hla) == 0 and len(allele_cols) > 0:
        heatmap_hla = hla_freq.head(10).index.tolist()

    hm_df = donor_hla[['DCP_ID', 'Ethnicity'] + heatmap_hla].copy()
    hm_df = hm_df[hm_df['Ethnicity'].isin(hm_eth_order)].copy()

    # column order: locus (A/B/C or DR/DQ/DP) -> frequency
    def locus_rank(x):
        x = str(x)
        if x.startswith('A'):
            return 0
        elif x.startswith('B'):
            return 1
        elif x.startswith('C'):
            return 2
        elif x.startswith('DR'):
            return 0
        elif x.startswith('DQ'):
            return 1
        elif x.startswith('DP'):
            return 2
        return 9

    col_order = sorted(heatmap_hla, key=lambda x: (locus_rank(x), -hm_df[x].sum(), str(x)))

    # row order: ethnicity -> row sum(desc) -> donor id
    hm_df['Ethnicity'] = pd.Categorical(hm_df['Ethnicity'], categories=hm_eth_order, ordered=True)
    hm_df['__n_hla__'] = hm_df[col_order].sum(axis=1)
    hm_df = hm_df.sort_values(['Ethnicity', '__n_hla__', 'DCP_ID'], ascending=[True, False, True]).copy()

    heatmap_mat = hm_df.set_index('DCP_ID')[col_order].astype(int)
    row_eth = hm_df['Ethnicity'].astype(str).tolist()

    # row label: ethnicity only (no donor id)
    row_labels = row_eth.copy()

    # ethnicity color strip
    row_colors = [eth_color_dict.get(e, '#cccccc') for e in row_eth]

    # ethnicity group boundary
    group_sizes = hm_df['Ethnicity'].value_counts(sort=False).reindex(hm_eth_order).fillna(0).astype(int)
    row_breaks = group_sizes.cumsum().iloc[:-1].tolist()

    # locus boundary
    col_breaks = []
    for i in range(1, len(col_order)):
        if locus_rank(col_order[i]) != locus_rank(col_order[i-1]):
            col_breaks.append(i)
        
    # -----------------------------
    # 6) summary strings
    # -----------------------------
    def join_or_none(xs, nmax=2):
        xs = list(xs)
        if len(xs) == 0:
            return 'None'
        if len(xs) <= nmax:
            return ' / '.join(xs)
        return ' / '.join(xs[:nmax]) + ' / ...'
    def is_valid_epitope(x):
        if x is None:
            return False
        s = str(x).strip().lower()
        return s not in ['', 'none', 'nan', 'na', 'unknown', 'ambiguous']

    if type == 'CD8':
        hla_text = f'{join_or_none(major_hla_A)}, {join_or_none(major_hla_B)}, {join_or_none(major_hla_C)}'
    else:
        hla_text = f'{join_or_none(major_hla_A)}, {join_or_none(major_hla_B)}, {join_or_none(major_hla_C)}'

    # predicted epitope text
    valid_species = [x for x in major_species if is_valid_epitope(x)]
    if len(valid_species) > 0:
        vdj_text = f'Predicted epitope: {join_or_none([truncate_text(x, 22) for x in valid_species], nmax=2)}'
    else:
        vdj_text = 'Predicted epitope: None'

    # matching / MHC text
    matched_vj_col  = 'matched_vj_seq'  if 'matched_vj_seq'  in clone_obs.columns else None
    matched_vdj_col = 'matched_vdj_seq' if 'matched_vdj_seq' in clone_obs.columns else None
    mhca_col        = 'mhc.a'           if 'mhc.a'           in clone_obs.columns else None

    matched_vj  = representative_value(clone_obs, matched_vj_col)
    matched_vdj = representative_value(clone_obs, matched_vdj_col)
    mhca_val    = representative_value(clone_obs, mhca_col)

    has_vj  = matched_vj is not None and str(matched_vj).strip().lower() not in ['nan', 'none', 'ambiguous', '']
    has_vdj = matched_vdj is not None and str(matched_vdj).strip().lower() not in ['nan', 'none', 'ambiguous', '']
    has_mhca = mhca_val is not None and str(mhca_val).strip().lower() not in ['nan', 'none', 'ambiguous', '']

    match_parts = []
    if len(valid_species) > 0:
        if has_vj and has_vdj:
            match_parts.append(
                'CDR3αβ matching'
            )
        elif has_vj:
            match_parts.append('CDR3α matching')
        elif has_vdj:
            match_parts.append('CDR3β matching')

        if has_mhca:
            match_parts.append(f'({mhca_val})')

    match_text = ' | '.join(match_parts)

    summary_text = f'Dominant HLA: {hla_text}  |  {vdj_text}'
    if match_text != '':
        summary_text += f'  |  {match_text}'

    # -----------------------------
    # 7) figure
    # -----------------------------
    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs = GridSpec(1, 2, width_ratios=[1.0, 1.8], wspace=0.08)

    ax_umap = fig.add_subplot(gs[0, 0])
    ax_ctx  = fig.add_subplot(gs[0, 1])
    ax_umap.set_box_aspect(0.95) 
    # ---- left: UMAP
    ax_umap.scatter(
        bg_df['UMAP1'], bg_df['UMAP2'],
        s=3, c='lightgray', alpha=0.22, linewidths=0, rasterized=True
    )
    for eth in Ethnicity_order:
        sub = plot_df[plot_df['highlight'] == eth]
        if len(sub) == 0:
            continue
        ax_umap.scatter(
            sub['UMAP1'], sub['UMAP2'],
            s=20, c=eth_color_dict[eth], label=eth, alpha=0.95, linewidths=0
        )

    ax_umap.set_title(f'Clone: {clone}', fontsize=16, pad=10)
    ax_umap.set_xlabel('UMAP1', fontsize=14)
    ax_umap.set_ylabel('UMAP2', fontsize=14)
    ax_umap.tick_params(labelsize=12)
    ax_umap.legend(frameon=False, bbox_to_anchor=(0.98, 0.98), loc='upper right',
                   fontsize=13, handlelength=0.8, handletextpad=0.3, borderpad=0.2)

    # ---- right: context
    ax_ctx.axis('off')
    ax_ctx.set_xlim(0, 1)
    ax_ctx.set_ylim(0, 1)

    n_cells = clone_obs.shape[0]
    n_donors = donor_origin['DCP_ID'].nunique()
    n_eth = donor_origin['Ethnicity'].nunique()

    # title block
#     ax_ctx.text(0.03, 0.98, 'Clone sequence and donor-level context',
#                 fontsize=18, fontweight='bold', ha='left')
#     ax_ctx.text(0.03, 0.91, clone, fontsize=17, ha='left', color='#333333')
    ax_ctx.text(0.03, 0.9,
                f'n cells = {n_cells}     n donors = {n_donors}     n ethnicities = {n_eth}',
                fontsize=17, ha='left', color='#444444')

    # sequence rows
    ax_ctx.text(0.03, 0.85, 'CDR3α', fontsize=17, fontweight='bold', ha='left')
    ax_ctx.text(0.15, 0.85, cdr3a if cdr3a is not None else 'NA', fontsize=17, ha='left')
    ax_ctx.text(0.52, 0.85, f'{trav if trav is not None else "NA"} / {traj if traj is not None else "NA"}',
                fontsize=17, ha='left', color='#222222')

    ax_ctx.text(0.03, 0.8, 'CDR3β', fontsize=17, fontweight='bold', ha='left')
    ax_ctx.text(0.15, 0.8, cdr3b if cdr3b is not None else 'NA', fontsize=17, ha='left')
    ax_ctx.text(0.52, 0.8, f'{trbv if trbv is not None else "NA"} / {trbj if trbj is not None else "NA"}',
                fontsize=17, ha='left', color='#222222')

    draw_summary_bar(ax_ctx, summary_text, x=0.03, y=0.67, w=0.93, h=0.065)

#     # -----------------------------
#     # right lower panel: HLA heatmap + pie
#     # -----------------------------
#     ax_ctx.text(0.03, 0.525, 'HLA alleles in clone-sharing donors',
#                 fontsize=19, fontweight='bold', ha='left')
#     ax_ctx.text(0.03, 0.488,
#                 'Rows are donors ordered by ethnicity; row labels show ethnicity only.',
#                 fontsize=13.5, ha='left', color='dimgray')

    # heatmap axes
#     ax_strip = ax_ctx.inset_axes([0.03, 0.1, 0.015, 0.40])
    ax_hm    = ax_ctx.inset_axes([0.03, 0.12, 0.6, 0.5])
    ax_pie   = ax_ctx.inset_axes([0.62, 0.09, 0.35, 0.50])

#     # left ethnicity strip
#     strip_rgba = np.array([mcolors.to_rgba(c) for c in row_colors]).reshape(len(row_colors), 1, 4)
#     ax_strip.imshow(strip_rgba, aspect='auto', interpolation='nearest')
#     ax_strip.set_xticks([])
#     ax_strip.set_yticks([])
#     for spine in ax_strip.spines.values():
#         spine.set_visible(False)

    # heatmap
    hm_cmap = ListedColormap([
        '#f8fafc',   # 0: very light blue-gray
        '#355c7d'    # 1: muted navy
    ])

    sns.heatmap(
        heatmap_mat,
        ax=ax_hm,
        cmap=hm_cmap,
        vmin=0, vmax=1,
        cbar=False,
        linewidths=0.8,
        linecolor='#dfe7ef',
        square=False
    )

    ax_hm.set_facecolor('#fbfcfe')
    ax_hm.set_xlabel('')
    ax_hm.set_ylabel('')

    ax_hm.set_yticks(np.arange(len(row_labels)) + 0.5)
    ax_hm.set_yticklabels(row_labels, rotation=0, fontsize=9)

    ax_hm.set_xticks(np.arange(len(col_order)) + 0.5)
    ax_hm.set_xticklabels(col_order, rotation=90, ha='right', fontsize=10)

    ax_hm.tick_params(axis='y', length=0, pad=2)
    ax_hm.tick_params(axis='x', length=0, pad=1)

    # row label color by ethnicity
    for tick, eth in zip(ax_hm.get_yticklabels(), row_eth):
        tick.set_color(eth_color_dict.get(eth, 'black'))
      
#     # ethnicity boundary lines
#     for y in row_breaks:
#         ax_hm.hlines(y, *ax_hm.get_xlim(), color='white', linewidth=2.8, zorder=5)
#         ax_strip.hlines(y - 0.5, -0.5, 0.5, color='white', linewidth=2.8)

    # locus boundary lines
    for x in col_breaks:
        ax_hm.vlines(x, *ax_hm.get_ylim(), color='#94a3b8', linewidth=2.2, zorder=5)

    # subtle outer frame
    for side in ['left', 'right', 'top', 'bottom']:
        ax_hm.spines[side].set_visible(True)
        ax_hm.spines[side].set_linewidth(0.8)
        ax_hm.spines[side].set_edgecolor('#cbd5e1')

    # -----------------------------
    # pie plot
    # -----------------------------
    ax_pie.set_title('Cell type proportion', fontsize=14, pad=8)

    ct_colors = [
        anno2_color_dict.get(ct, '#cccccc') if ct != 'Other' else '#d9d9d9'
        for ct in celltype_plot.index
    ]

    def autopct_func(pct):
        return f'{pct:.0f}%' if pct >= 8 else ''

    pie_labels = [
        k if (v / celltype_plot.sum()) >= 0.12 else ''
        for k, v in celltype_plot.items()
    ]

    wedges, texts, autotexts = ax_pie.pie(
        celltype_plot.values,
        labels=pie_labels,
        colors=ct_colors,
        startangle=90,
        counterclock=False,
        autopct=autopct_func,
        pctdistance=0.72,
        labeldistance=1.08,
        wedgeprops=dict(width=0.6, edgecolor='k',
                       alpha=0.7))
    for t in texts:
        t.set_fontsize(8.5)
    for t in autotexts:
        t.set_fontsize(8.5)

    ax_pie.axis('equal')
  
#     # footnote
#     ax_ctx.text(
#         0.03, 0.01,
#         'Interpretation guide: left panel shows clone localization on the UMAP; '
#         'right panel summarizes sequence, shared-donor HLA context, and a conceptual HLA-to-ethnicity flow.',
#         fontsize=11.5, color='dimgray', ha='left', va='bottom'
#     )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')

    if show:
        plt.show()
#     plt.close(fig)

In [ ]:
def plot_clone_junction_logo(vvdata, clone_id, clone_id_col='clone_id',
                             vdj_col='junction_aa_VDJ', vj_col='junction_aa_VJ',
                             figsize=(14, 3)):
    sub = vvdata.obs.loc[vvdata.obs[clone_id_col].astype(str) == str(clone_id)]
    if sub.empty:
        print(f"[!] no cells found for clone_id={clone_id}.")
        return None

    fig, axes = plt.subplots(2, 1, figsize=(figsize[0], figsize[1] * 2))

    for ax, col in zip(axes, [vdj_col, vj_col]):
        seqs = sub[col].dropna().astype(str).tolist()
        seqs = [s for s in seqs if s and s.lower() != 'nan']

        if not seqs:
            ax.set_title(f"{col} | clone_id={clone_id} — no sequences")
            ax.axis('off')
            continue

        # assume identical lengths; if mixed, keep only the most common
        lens = pd.Series([len(s) for s in seqs])
        dom_len = lens.mode().iloc[0]
        kept = [s for s in seqs if len(s) == dom_len]

        mat = logomaker.alignment_to_matrix(sequences=kept, to_type='information')
        logomaker.Logo(mat, ax=ax, color_scheme='chemistry')
        ax.set_title(f"{col} | clone_id={clone_id} (n={len(kept)}, len={dom_len})")
        ax.set_ylabel('Probability')
        ax.set_xlabel('Position')

    plt.tight_layout()
    return fig   # <- return fig only (no plt.show())

### Render representative public clones featured in Fig. 2G

In [ ]:
# Representative clones reported in Fig. 2G
# (clone_id_85: clone IDs computed at 0.85 Hamming-quasi-public threshold)
# Replace with the clones of interest from your own analysis.
representative_clones = [
    ('abT_549_3_25_466_7_91', 'CD8'),
    ('abT_143_5_115_1178_10_59', 'CD8'),
    ('abT_393_7_263_1548_4_5', 'CD8'),
    ('abT_416_8_219_291_5_33', 'CD4'),
]

for cid, ctype in representative_clones:
    fig = plot_clone_junction_logo(vvdata, clone_id_col='clone_id_85', clone_id=cid)
    if fig is not None:
        fig.savefig(f'figures/Fig2_{cid}_logo.pdf', dpi=300, bbox_inches='tight')
        plt.show()
    plot_clone_passport(cid, vvdata, hladf, type=ctype)
